In [1]:
!pip install faiss-cpu open_clip_torch accelerate qwen-vl-utils evaluate rouge_score bert_score scikit-learn -q

In [2]:
import os
import torch
import time
import evaluate
import numpy as np
import gc
import random
import faiss
import open_clip
import torch.nn as nn
import transformers
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import ImageFile
from transformers import AutoModel, AutoTokenizer

# --- 1. CẤU HÌNH & VÁ LỖI KIẾN TRÚC ---
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
ImageFile.LOAD_TRUNCATED_IMAGES = True

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Đường dẫn Faiss Index và Metadata Text-only
INDEX_PATH = "/kaggle/input/datasets/trananhvu1412/pathvqa-text-only/pathvqa_textonly.index"
META_PATH = "/kaggle/input/datasets/trananhvu1412/pathvqa-text-only/pathvqa_textonly_metadata.pt"


# --- 2. RAG RETRIEVER (TEXT-ONLY) ---
class PathVqaTextOnlyRetriever:
    def __init__(self):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        
        # Tải Faiss Index và Metadata
        self.faissIndex = faiss.read_index(INDEX_PATH)
        self.metaDataList = torch.load(META_PATH, weights_only=False)
        
        # Chỉ tải Text Encoder của BioMedCLIP (Không cần Image Encoder & Gated Fusion)
        modelName = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        self.bioMedClip, _, _ = open_clip.create_model_and_transforms(modelName)
        self.tokenizer = open_clip.get_tokenizer(modelName)
        self.bioMedClip.to(self.device).eval()

    def retrieveTopK(self, questionText, topK=2): 
        with torch.inference_mode():
            # Tokenize và Encode trực tiếp từ chuỗi câu hỏi
            tokens = self.tokenizer([questionText]).to(self.device)
            
            txtFeat = self.bioMedClip.encode_text(tokens)
            txtFeat /= txtFeat.norm(dim=-1, keepdim=True)

            queryVec = txtFeat.cpu().numpy().astype('float32')
            distances, indices = self.faissIndex.search(queryVec, topK)
            
            return [self.metaDataList[idx] for idx in indices[0]], distances[0]


# --- 3. MÔ HÌNH SUY LUẬN INTERNVL RAG ---
class InternVLRagGenerator:
    def __init__(self):
        torch.cuda.empty_cache()
        gc.collect()
        
        print("\n[Hạ tầng] Đang khởi tạo mô hình InternVL3.5-2B (Text-Only RAG K=2)...")
        modelId = "OpenGVLab/InternVL3_5-2B-Instruct"
        
        self.model = AutoModel.from_pretrained(
            modelId,
            torch_dtype=torch.bfloat16,
            trust_remote_code=True,
            device_map="auto"
        ).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
        print("✅ Đã nạp thành công mô hình InternVL3.5-2B!")

    def processImage(self, pilImg, targetSize=448):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        imageTransform = T.Compose([
            T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
        ])
        return imageTransform(pilImg).unsqueeze(0)

    def generate(self, testImg, testQuest, refContexts):
        # 💡 Tích hợp cấu trúc Prompt khắt khe
        promptText = (
            "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
            "based on an input pathology image and retrieved similar cases.\n"
            "Strictly follow these rules:\n"
            "1. Analyze the provided [Target Case Image] and [Current Question].\n"
            "2. Evaluate the [Retrieved Evidence] from the knowledge base.\n"
            "3. Provide a short, concise, and definitive answer (usually 1 to 5 words).\n"
            "4. Do NOT explain your reasoning.\n\n"
            "[Retrieved Evidence (Top-3)]\n"
            "Below are similar past cases from the clinical database:\n\n"
        )
        
        # Chỉ nạp thông tin dạng văn bản (Text-only RAG)
        for i, ctx in enumerate(refContexts):
            promptText += f"Case {i+1}:\nQuestion: {ctx['question']}\nAnswer: {ctx['answer']}\n\n"
                
        # Chỉ xử lý và truyền 1 ảnh duy nhất (Ảnh mục tiêu)
        targetPixelValues = self.processImage(testImg)
        pixelValuesList = [targetPixelValues]
        
        promptText += (
            f"[Target Case Image]\n<image>\n\n"
            f"Current Question: {testQuest}\n\n"
            "Based on the visual evidence and the clinical context provided above, what is the correct answer for the current case?\n"
            "Final Answer: "
        )
        
        stackedPixelValues = torch.cat(pixelValuesList, dim=0).to(torch.bfloat16).cuda()
        generationConfig = dict(max_new_tokens=15, do_sample=False)
        
        with torch.inference_mode():
            response = self.model.chat(
                tokenizer=self.tokenizer,
                pixel_values=stackedPixelValues,
                question=promptText,
                generation_config=generationConfig
            )
            
        return response.strip()


# --- 4. HÀM THỰC THI CHÍNH ---
def runInternVLTextOnlyRagPathVqaBenchmark(sampleSize=500):
    print("1. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("2. Tải Retriever và Generator...")
    retriever = PathVqaTextOnlyRetriever()
    slm = InternVLRagGenerator()
    
    print("3. Tải bộ dữ liệu PathVQA...")
    testDS = load_dataset("flaviagiammarino/path-vqa", split="test")
    
    totalIndices = len(testDS)
    sampledIndices = random.sample(range(totalIndices), min(sampleSize, totalIndices))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    skippedCount = 0
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM TEXT-ONLY RAG (INTERNVL3.5-2B | K=5) TRÊN {len(sampledIndices)} MẪU PATHVQA...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()

    for progressCount, idx in enumerate(sampledIndices):
        try:
            item = testDS[idx]
            
            if item.get('image') is None:
                skippedCount += 1
                continue
                
            rawImg = item['image'].convert("RGB")
            questionText = str(item['question'])
            
            startTimeSample = time.time()
            
            # 💡 Chỉ truyền Text vào hàm retrieveTopK (Không truyền rawImg)
            contexts, _ = retriever.retrieveTopK(questionText, topK=2)
            
            aiAnswerRaw = slm.generate(rawImg, questionText, contexts)
            
        except Exception as e:
            print(f"⚠️ Lỗi ngầm tại mẫu {idx}: {e}")
            skippedCount += 1
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 20 == 0: 
            print(f"Tiến độ: {progressCount + 1}/{len(sampledIndices)} mẫu...")

        del rawImg, contexts, aiAnswerRaw
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    endTotalTime = time.time()
    
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM - TEXT-ONLY RAG (INTERNVL3.5-2B ON PATHVQA | K=2)\n" + "="*60)
    print(f"📌 Đã quét: {len(sampledIndices)} | Bỏ qua do lỗi: {skippedCount}")
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"\n[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    if latencyList:
        print("\n[3. Computational Efficiency Measurements]")
        print(f"  - Inference Latency (Độ trễ trung bình) : {np.mean(latencyList):.2f} s/mẫu")
        print(f"  - Throughput (Tốc độ xử lý)             : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}        : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runInternVLTextOnlyRagPathVqaBenchmark(sampleSize=500)

1. Khởi tạo các thư viện đo lường (Metrics)...


2. Tải Retriever và Generator...


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  784MB            

open_clip_pytorch_model.bin: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

... (56 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  - BLEU-1   : 0.0950
  - BLEU-2   : 0.0915
  - BLEU-3   : 0.0865
  - ROUGE-L  : 0.0797
  - BERTScore: 0.8561

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ trung bình) : 1.25 s/mẫu
  - Throughput (Tốc độ xử lý)             : 0.64 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0        : 4167.25 MB
  - Peak Memory Usage on GPU 1        : 2265.34 MB
